# 低通滤波器对方波的影响

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from ipywidgets import interact, FloatSlider, IntSlider


# ----- 设置中文字体，避免警告 -----
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'KaiTi', 'Arial Unicode MS']  # 按优先级尝试
plt.rcParams['axes.unicode_minus'] = False  # 正确显示负号
# ---------------------------------

# 傅里叶级数：方波合成原理与数据解析

傅里叶级数（Fourier Series）证明了周期函数可以用正弦和余弦函数的无穷级数来表示。对于一个理想的方波，其合成过程主要是奇次谐波的叠加。

---

## 1. 数学表达公式

方波 $f(t)$ 的展开式如下：

$$f(t) = \frac{4}{\pi} \sum_{n=1,3,5,\dots}^{\infty} \frac{1}{n} \sin(n \omega t)$$



### 数据项定义：
- **$n$**: 谐波次数（仅取 $1, 3, 5, \dots$ 等奇数）。
- **$\frac{4}{\pi n}$**: 第 $n$ 次谐波的振幅。随着 $n$ 增大，振幅按 $1/n$ 衰减。
- **$\omega$**: 基频角速度，决定了方波的周期。

---

## 2. 逼近过程的数据对比表

随着叠加项数 $N$ 的增加，波形从圆润的正弦波逐渐向直角的方波演变：

| 叠加项数 (N) | 最高谐波次数 | 振幅衰减 (1/n) | 波形特征描述 |
| :--- | :--- | :--- | :--- |
| 1 | 1 | 1.000 | 基础正弦波，波峰约 1.27 |
| 2 | 3 | 0.333 | 顶部变平，开始出现方波轮廓 |
| 5 | 9 | 0.111 | 边缘变陡，顶部震荡频率增加 |
| 50 | 99 | 0.010 | 极度接近方波，仅在跳变沿有毛刺 |

---

## 3. 关键物理现象：吉布斯现象 (Gibbs Phenomenon)

在方波的间断点（跳变沿），即使 $N$ 趋于无穷大，合成波形的峰值总会超过原方波幅值的 **9%** 左右。



- **原因**：使用连续函数（正弦波）去拟合不连续点（方波跳变）时，在跳变点附近产生的能量集中。
- **数据表现**：在代码模拟中，你会发现 $N$ 越大，这个“毛刺”越细，但高度几乎不变。

---

## 4. Python 数据生成逻辑示例

In [ ]:
t=np.linspace(-np.pi * 3,np.pi * 3,1000)

def square_series(N=1):

    y=np.zeros_like(t)

    for n in range(1,N+1):
        y+=np.sin((2*n-1)*t)/(2*n-1)

    y*=4/np.pi

    plt.figure(figsize=(8,4))
    plt.plot(t,y)
    plt.title(f'Fourier Series Approx N={N}')
    plt.grid()
    plt.show()

interact(square_series,N=IntSlider(min=1,max=100,step=1,value=1))

# 不同低通滤波器对方波的影响

In [ ]:
# 参数设置
fs = 1000               # 采样率 (Hz)
T = 1.0                 # 总时间 (秒)
N = int(fs * T)         # 采样点数
t = np.linspace(0, T, N, endpoint=False)

# 生成一个矩形脉冲（模拟一个符号）
pulse_start = 0.2       # 起始时间 (秒)
pulse_end = 0.3         # 结束时间 (秒)
x = np.zeros(N)
x[(t >= pulse_start) & (t < pulse_end)] = 1.0

# 理想低通滤波对方波脉冲的影响：频域与时域交互可视化

没有画出滤波器响应，这是理想低通滤波器和实际滤波器实现方式的本质区别：理想低通滤波器不是通过滤波器系数实现的，而是直接在频域乘以一个矩形窗



理想低通滤波器的频率响应：

H(f) = 1   |f| ≤ fc  
H(f) = 0   |f| > fc

其时域冲激响应为：

h(t) = sinc(2πfc t)

因此在时域会产生明显的 **sinc 振铃 (ringing)**。

In [ ]:
def plot_ideal_filtered(fc):

    # 计算频谱
    X = np.fft.fft(x)
    freqs_full = np.fft.fftfreq(N, 1/fs)

    # 构造理想低通滤波器响应
    H = np.zeros_like(freqs_full)
    H[np.abs(freqs_full) <= fc] = 1

    # 频域滤波
    Y = X * H

    # 回到时域
    y = np.real(np.fft.ifft(Y))

    # 只取正频率部分用于绘图
    freqs = freqs_full[:N//2]
    Xp = X[:N//2]
    Yp = Y[:N//2]
    Hp = H[:N//2]

    fig, (ax1, ax2) = plt.subplots(2,1,figsize=(8,8))

    # 频域
    ax1.plot(freqs, np.abs(Xp), label="原始信号频谱", alpha=0.7)
    ax1.plot(freqs, np.abs(Yp), label="Ideal LPF 滤波后频谱", alpha=0.7)

    scale = np.max(np.abs(Xp)) / np.max(Hp)
    ax1.plot(freqs, Hp*scale, 'k--', label="Ideal LPF 响应")

    ax1.axvline(fc, color='r', linestyle=':', label=f'fc={fc} Hz')

    ax1.set_xlim(0,200)
    ax1.set_xlabel("频率 (Hz)")
    ax1.set_ylabel("幅度")
    ax1.legend()
    ax1.grid(True)
    ax1.set_title("频域：Ideal低通滤波")

    # 时域
    ax2.plot(t, x, label="原始脉冲")
    ax2.plot(t, y, label="Ideal LPF 滤波后脉冲")

    ax2.set_xlim(0,0.6)
    ax2.set_xlabel("时间 (秒)")
    ax2.set_ylabel("幅度")
    ax2.legend()
    ax2.grid(True)
    ax2.set_title("时域：Ideal LPF后的脉冲")

    plt.tight_layout()
    plt.show()


fc_slider = FloatSlider(value=50, min=1, max=200, step=1, description='截止频率 (Hz)')
interact(plot_ideal_filtered, fc=fc_slider)

# Butterworth低通滤波对方波脉冲的影响：频域与时域交互可视化

本 Notebook 通过滑块调节低通滤波器的截止频率，实时展示：
- 原始脉冲与滤波后脉冲的频谱对比
- 原始脉冲与滤波后脉冲的时域波形（展宽现象）

拖动滑块，观察截止频率降低时，高频分量被削减，时域脉冲如何逐渐展宽并产生拖尾。

In [ ]:
def plot_filtered(fc):
    # 设计二阶巴特沃斯低通滤波器
    b, a = signal.butter(2, fc/(fs/2), btype='low')
    y = signal.filtfilt(b, a, x)   # 零相位滤波，避免相位偏移
    
    # 计算频谱
    X = np.fft.fft(x)
    Y = np.fft.fft(y)
    freqs = np.fft.fftfreq(N, 1/fs)[:N//2]
    
    # 计算滤波器频率响应
    w, h = signal.freqz(b, a, worN=8000, fs=fs)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 8))
    
    # 频域子图
    ax1.plot(freqs, np.abs(X[:N//2]), label='原始信号频谱', alpha=0.7)
    ax1.plot(freqs, np.abs(Y[:N//2]), label='滤波后信号频谱', alpha=0.7)
    scale = np.max(np.abs(X[:N//2])) / np.max(np.abs(h))
    ax1.plot(w, np.abs(h) * scale, 'k--', label='滤波器响应（缩放）')
    ax1.axvline(fc, color='r', linestyle=':', label=f'截止频率 fc = {fc} Hz')
    ax1.set_xlim(0, 200)
    ax1.set_xlabel('频率 (Hz)')
    ax1.set_ylabel('幅度')
    ax1.legend(loc='upper right')
    ax1.grid(True)
    ax1.set_title('频域：原始与滤波后频谱对比')
    
    # 时域子图
    ax2.plot(t, x, label='原始脉冲')
    ax2.plot(t, y, label='滤波后脉冲')
    ax2.set_xlim(0, 0.6)
    ax2.set_xlabel('时间 (秒)')
    ax2.set_ylabel('幅度')
    ax2.legend()
    ax2.grid(True)
    ax2.set_title('时域：脉冲展宽与拖尾')
    
    plt.tight_layout()
    plt.show()

fc_slider = FloatSlider(value=50, min=1, max=200, step=1, description='截止频率 (Hz)')
interact(plot_filtered, fc=fc_slider)

# Bessel低通滤波对方波脉冲的影响：频域与时域交互可视化

In [ ]:
def plot_bessel_filtered(fc):

    # 设计 Bessel 低通滤波器
    b, a = signal.bessel(2, fc/(fs/2), btype='low', norm='phase')

    # 滤波
    y = signal.filtfilt(b, a, x)

    # 计算频谱
    X = np.fft.fft(x)
    Y = np.fft.fft(y)
    freqs = np.fft.fftfreq(N, 1/fs)[:N//2]

    # 滤波器频率响应
    w, h = signal.freqz(b, a, worN=8000, fs=fs)

    fig, (ax1, ax2) = plt.subplots(2,1,figsize=(8,8))

    # 频域
    ax1.plot(freqs, np.abs(X[:N//2]), label="原始信号频谱", alpha=0.7)
    ax1.plot(freqs, np.abs(Y[:N//2]), label="Bessel滤波后频谱", alpha=0.7)

    scale = np.max(np.abs(X[:N//2])) / np.max(np.abs(h))
    ax1.plot(w, np.abs(h)*scale, 'k--', label="Bessel滤波器响应")

    ax1.axvline(fc, color='r', linestyle=':', label=f'fc={fc} Hz')

    ax1.set_xlim(0,200)
    ax1.set_xlabel("频率 (Hz)")
    ax1.set_ylabel("幅度")
    ax1.legend()
    ax1.grid(True)
    ax1.set_title("频域：Bessel低通滤波")

    # 时域
    ax2.plot(t, x, label="原始脉冲")
    ax2.plot(t, y, label="Bessel滤波后脉冲")

    ax2.set_xlim(0,0.6)
    ax2.set_xlabel("时间 (秒)")
    ax2.set_ylabel("幅度")
    ax2.legend()
    ax2.grid(True)
    ax2.set_title("时域：Bessel滤波后的脉冲")

    plt.tight_layout()
    plt.show()
    
fc_slider = FloatSlider(value=50, min=1, max=200, step=1, description='截止频率 (Hz)')
interact(plot_bessel_filtered, fc=fc_slider)

# Chebyshev 低通滤波对方波脉冲的影响：频域与时域交互可视化

In [ ]:
def plot_chebyshev_filtered(fc):

    ripple = 1   # 通带波纹 (dB)

    # 设计 Chebyshev Type I 低通滤波器
    b, a = signal.cheby1(2, ripple, fc/(fs/2), btype='low')

    # 滤波
    y = signal.filtfilt(b, a, x)

    # 计算频谱
    X = np.fft.fft(x)
    Y = np.fft.fft(y)
    freqs = np.fft.fftfreq(N, 1/fs)[:N//2]

    # 计算滤波器频率响应
    w, h = signal.freqz(b, a, worN=8000, fs=fs)

    fig, (ax1, ax2) = plt.subplots(2,1,figsize=(8,8))

    # 频域
    ax1.plot(freqs, np.abs(X[:N//2]), label="原始信号频谱", alpha=0.7)
    ax1.plot(freqs, np.abs(Y[:N//2]), label="Chebyshev滤波后频谱", alpha=0.7)

    scale = np.max(np.abs(X[:N//2])) / np.max(np.abs(h))
    ax1.plot(w, np.abs(h)*scale, 'k--', label="Chebyshev滤波器响应")

    ax1.axvline(fc, color='r', linestyle=':', label=f'fc={fc} Hz')

    ax1.set_xlim(0,200)
    ax1.set_xlabel("频率 (Hz)")
    ax1.set_ylabel("幅度")
    ax1.legend()
    ax1.grid(True)
    ax1.set_title("频域：Chebyshev低通滤波")

    # 时域
    ax2.plot(t, x, label="原始脉冲")
    ax2.plot(t, y, label="Chebyshev滤波后脉冲")

    ax2.set_xlim(0,0.6)
    ax2.set_xlabel("时间 (秒)")
    ax2.set_ylabel("幅度")
    ax2.legend()
    ax2.grid(True)
    ax2.set_title("时域：Chebyshev滤波后的脉冲")

    plt.tight_layout()
    plt.show()

fc_slider = FloatSlider(value=50, min=1, max=200, step=1, description='截止频率 (Hz)')
interact(plot_chebyshev_filtered, fc=fc_slider)

# 四种滤波器对比
| 滤波器         | 频域特点 | 时域特点      | SerDes意义 |
| ----------- | ---- | --------- | -------- |
| Ideal       | 突变截止 | 强 sinc 振铃 | 理论模型     |
| Butterworth | 最平滑  | 边沿变慢      | 模拟带宽     |
| Bessel      | 相位线性 | 波形保持好     | 测量系统     |
| Chebyshev   | 截止最陡 | 振铃明显      | 高选择性     |

频域越陡
↓
时域振铃越强


## lfilter 与 filtfilt

在 `scipy.signal` 中常用的两种滤波函数：

- `signal.lfilter`
- `signal.filtfilt`

它们的主要区别是 **是否产生相位延迟**。

---

### 1. lfilter

`lfilter` 是 **标准数字滤波器实现**：

\[
y[n] = \sum b_k x[n-k] - \sum a_k y[n-k]
\]

特点：

- 因果系统（Causal）
- 输出只依赖当前和过去输入
- 存在相位延迟
- 可用于实时系统

时域表现：

```
输入:
      ______
_____|      |_____

输出:
      ______
     /      \
____/        \____
               ~~~~
```

下降沿会出现 **拖尾（ISI）**。

物理意义：

```
真实信道 ≈ lfilter
```

例如：

- PCB 走线
- 电缆
- 模拟滤波器

---

### 2. filtfilt

`filtfilt` 的方法：

```
前向滤波 → 信号反转 → 再滤波 → 再反转
```

特点：

- 零相位滤波
- 无时间延迟
- 输出波形对称

时域表现：

```
      ______
     /      \
____/        \____
```

但它是：

```
非因果系统
```

因为输出依赖未来数据，所以：

```
不能用于实时系统
```

适合：

- 离线信号处理
- 数据分析

---

### 3. 对比

| 特性 | lfilter | filtfilt |
|-----|--------|---------|
|系统类型|因果系统|非因果系统|
|相位延迟|有|无|
|实时系统|支持|不支持|
|真实信道|符合|不符合|

---

### 总结

```
lfilter  → 真实物理系统 / 信道模拟
filtfilt → 零相位滤波 / 数据分析
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
from ipywidgets import interact, FloatSlider, Checkbox, HBox

# -----------------------------
# 信号参数
# -----------------------------
fs = 1000
T = 1
N = int(fs*T)

t = np.linspace(0, T, N, endpoint=False)

# 矩形脉冲
x = np.zeros(N)
x[(t >= 0.2) & (t < 0.3)] = 1


# -----------------------------
# Ideal LPF
# -----------------------------
def ideal_lpf(signal_in, fc):

    X = fft(signal_in)
    freqs = fftfreq(N, 1/fs)

    H = np.zeros(N)
    H[np.abs(freqs) <= fc] = 1

    Y = X * H
    y = np.real(np.fft.ifft(Y))

    return y


# -----------------------------
# 主绘图函数
# -----------------------------
def compare_filters(fc, show_lfilter=True, show_filtfilt=True):

    X = fft(x)
    freqs = fftfreq(N, 1/fs)

    # Butterworth
    b1, a1 = signal.butter(2, fc/(fs/2))
    y_butter_l = signal.lfilter(b1, a1, x)
    y_butter_f = signal.filtfilt(b1, a1, x)

    # Bessel
    b2, a2 = signal.bessel(2, fc/(fs/2), norm='phase')
    y_bessel_l = signal.lfilter(b2, a2, x)
    y_bessel_f = signal.filtfilt(b2, a2, x)

    # Chebyshev
    b3, a3 = signal.cheby1(2, 1, fc/(fs/2))
    y_cheby_l = signal.lfilter(b3, a3, x)
    y_cheby_f = signal.filtfilt(b3, a3, x)

    # Ideal
    y_ideal = ideal_lpf(x, fc)

    filters = [
        ("Ideal", y_ideal, y_ideal),
        ("Butterworth", y_butter_l, y_butter_f),
        ("Bessel", y_bessel_l, y_bessel_f),
        ("Chebyshev", y_cheby_l, y_cheby_f),
    ]

    fig, axes = plt.subplots(4,2, figsize=(12,12))

    for i,(name, yl, yf) in enumerate(filters):

        Yl = np.abs(fft(yl))[:N//2]
        Yf = np.abs(fft(yf))[:N//2]

        # -------- 频域 --------
        axes[i,0].plot(freqs[:N//2], np.abs(X[:N//2]), label="Original", alpha=0.5)

        if show_lfilter:
            axes[i,0].plot(freqs[:N//2], Yl, label="lfilter")

        if show_filtfilt:
            axes[i,0].plot(freqs[:N//2], Yf, label="filtfilt")

        axes[i,0].set_xlim(0,200)
        axes[i,0].set_title(f"{name} Spectrum")
        axes[i,0].grid(True)
        axes[i,0].legend()

        # -------- 时域 --------
        axes[i,1].plot(t, x, label="Original", alpha=0.5)

        if show_lfilter:
            axes[i,1].plot(t, yl, label="lfilter")

        if show_filtfilt:
            axes[i,1].plot(t, yf, label="filtfilt")

        axes[i,1].set_xlim(0,0.6)
        axes[i,1].set_title(f"{name} Time Domain")
        axes[i,1].grid(True)
        axes[i,1].legend()

    plt.tight_layout()
    plt.show()


# -----------------------------
# 交互控件
# -----------------------------
fc_slider = FloatSlider(value=50, min=5, max=200, step=1, description='fc')

lfilter_box = Checkbox(value=True, description='Show lfilter')
filtfilt_box = Checkbox(value=True, description='Show filtfilt')

interact(compare_filters,
         fc=fc_slider,
         show_lfilter=lfilter_box,
         show_filtfilt=filtfilt_box)